In [ ]:
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / 'utils').is_dir()
            and (candidate / 'vignettes').is_dir()
            and (candidate / 'README.md').exists()
        ):
            return candidate
    raise FileNotFoundError('Could not locate eQTL_annotations_for_susine project root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.paths import find_gtex_parquet, get_project_paths

PATHS = get_project_paths(PROJECT_ROOT)

# =============================================================================
# USER CONFIGURATION
# =============================================================================
GENE_NAME = 'CBX8'
GENE_ID = 'ENSG00000141570.11'
GTEX_TISSUE = 'Lung'
GTEX_CHROM = 'chr17'
PARQUET_PATH = find_gtex_parquet(tissue=GTEX_TISSUE, chrom=GTEX_CHROM, anchor=PROJECT_ROOT)

# Filtering Parameters
MAF_MIN = 0.01
MAF_MAX = 0.99
MIN_SAMPLE_SIZE = 50

# Borzoi Model Parameters (standard values)
SEQ_LEN = 524288
LABEL_LEN = 196608

# =============================================================================
# LOAD DATA
# =============================================================================

print(f'Project root: {PROJECT_ROOT}')
print(f'Loading data for {GENE_NAME} ({GENE_ID}) from {PARQUET_PATH}...')
table = pq.read_table(
    PARQUET_PATH,
    filters=[('gene_id', '==', GENE_ID)],
)
df = table.to_pandas()

print(f'Loaded {len(df)} SNPs')
print(f'Columns: {df.columns.tolist()}')
df.head()


In [ ]:
# Add sample_size column
df['sample_size'] = df['ma_count'] / (2 * df['af'])

# =============================================================================
# APPLY FILTERS
# =============================================================================
print(f"Filtering variants...")
raw_variant_count = len(df)
print(f"  Initial count: {raw_variant_count}")

# MAF Filter
df = df[(df['af'] >= MAF_MIN) & (df['af'] <= MAF_MAX)].copy()
post_af_count = len(df)
print(f"  After MAF filter ({MAF_MIN} <= AF <= {MAF_MAX}): {post_af_count}")

# Sample Size Filter
df = df[df['sample_size'] > MIN_SAMPLE_SIZE].copy()
post_sample_size_count = len(df)
print(f"  After Sample Size filter (>{MIN_SAMPLE_SIZE}): {post_sample_size_count}")

# =============================================================================
# STATISTICS
# =============================================================================

# Define thresholds in base pairs
tss_threshold_100kb = 100_000
tss_threshold_1mb = 1_000_000

# Calculate statistics for 100KB radius
snps_within_100kb = len(df[df['tss_distance'].abs() <= tss_threshold_100kb])
snps_outside_100kb = len(df[df['tss_distance'].abs() > tss_threshold_100kb])
sig_snps_within_100kb = len(df[(df['tss_distance'].abs() <= tss_threshold_100kb) & (df['pval_nominal'] < 0.05)])
sig_snps_outside_100kb = len(df[(df['tss_distance'].abs() > tss_threshold_100kb) & (df['pval_nominal'] < 0.05)])

# Calculate statistics for 1MB radius
total_snps = len(df)
snps_within_1mb = len(df[df['tss_distance'].abs() <= tss_threshold_1mb])
snps_outside_1mb = total_snps - snps_within_1mb
sig_snps_within_1mb = len(df[(df['tss_distance'].abs() <= tss_threshold_1mb) & (df['pval_nominal'] < 0.05)])
sig_snps_outside_1mb = len(df[(df['tss_distance'].abs() > tss_threshold_1mb) & (df['pval_nominal'] < 0.05)])

# Calculate average effective sample sizes
avg_sample_size_all = df['sample_size'].mean()
avg_sample_size_100kb = df[df['tss_distance'].abs() <= tss_threshold_100kb]['sample_size'].mean()
avg_sample_size_1mb = df[df['tss_distance'].abs() <= tss_threshold_1mb]['sample_size'].mean()

print(f"\n=== {GENE_NAME} eQTL Analysis (Filtered) ===\n")
print(f"Total SNPs: {total_snps}")
print(f"Average effective sample size (all SNPs): {avg_sample_size_all:.1f}")

print(f"\n--- 100KB Radius ---")
print(f"SNPs within 100KB of TSS: {snps_within_100kb}")
print(f"SNPs outside 100KB of TSS: {snps_outside_100kb}")
print(f"Significant SNPs (pval_nominal < 0.05):")
print(f"  - Within 100KB of TSS: {sig_snps_within_100kb}")
print(f"  - Outside 100KB of TSS: {sig_snps_outside_100kb}")
print(f"Significance rate within 100KB: {sig_snps_within_100kb/snps_within_100kb*100:.1f}%")
print(f"Average effective sample size (within 100KB): {avg_sample_size_100kb:.1f}")

print(f"\n--- 1MB Radius ---")
print(f"SNPs within 1MB of TSS: {snps_within_1mb}")
print(f"SNPs outside 1MB of TSS: {snps_outside_1mb}")
print(f"Significant SNPs (pval_nominal < 0.05):")
print(f"  - Within 1MB of TSS: {sig_snps_within_1mb}")
print(f"  - Outside 1MB of TSS: {sig_snps_outside_1mb}")
print(f"Significance rate within 1MB: {sig_snps_within_1mb/snps_within_1mb*100:.1f}%")
print(f"Average effective sample size (within 1MB): {avg_sample_size_1mb:.1f}")

In [ ]:
import gzip
import urllib.request
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd

from utils.paths import get_project_paths

PATHS = get_project_paths(PROJECT_ROOT)

# =============================================================================
# UTILITY FUNCTIONS (Copied from utils/data_ingestion.py to avoid torch dependency)
# =============================================================================

def ensure_dir(path: Path) -> Path:
    """Create a directory if it does not exist and return the Path."""
    path.mkdir(parents=True, exist_ok=True)
    return path


def download_gtf_if_needed(cache_dir: Path, genome: str = 'hg38') -> Path:
    """Download the GENCODE GTF file if it is not already available locally."""
    cache_dir = ensure_dir(Path(cache_dir))
    gtf_path = cache_dir / f'{genome}_gencode.gtf.gz'

    if gtf_path.exists():
        print(f'GTF file already exists: {gtf_path}')
        return gtf_path

    print('Downloading GENCODE GTF file...')
    url = 'https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/gencode.v44.annotation.gtf.gz'
    urllib.request.urlretrieve(url, gtf_path)
    print(f'Downloaded GTF to: {gtf_path}')
    return gtf_path


def _parse_gtf_for_gene(gtf_path: Path, gene_name: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Stream through the GTF and retain only entries for the requested gene."""
    genes = []
    exons = []
    gene_key = gene_name.upper()
    opener = gzip.open if str(gtf_path).endswith('.gz') else open

    with opener(gtf_path, 'rt') as handle:
        for line in handle:
            if not line or line.startswith('#'):
                continue

            fields = line.strip().split('	')
            if len(fields) < 9:
                continue

            chrom, source, feature, start, end, score, strand, frame, attributes = fields

            attr_dict: Dict[str, str] = {}
            for attr in attributes.split(';'):
                attr = attr.strip()
                if not attr:
                    continue
                parts = attr.split(' ', 1)
                if len(parts) == 2:
                    key, value = parts
                    attr_dict[key] = value.strip('"')

            attr_gene_name = attr_dict.get('gene_name', '')
            if attr_gene_name.upper() != gene_key:
                continue

            if feature == 'gene':
                genes.append({
                    'chrom': chrom,
                    'start': int(start) - 1,
                    'end': int(end),
                    'gene_name': attr_gene_name,
                    'gene_type': attr_dict.get('gene_type', ''),
                    'strand': strand,
                })
            elif feature == 'exon':
                exons.append({
                    'chrom': chrom,
                    'start': int(start) - 1,
                    'end': int(end),
                    'gene_name': attr_gene_name,
                    'strand': strand,
                })

    genes_df = pd.DataFrame(genes)
    exons_df = pd.DataFrame(exons)

    if genes_df.empty:
        raise ValueError(f"Gene '{gene_name}' not found in annotations at {gtf_path}")

    genes_df['tss'] = genes_df.apply(
        lambda row: row['start'] if row['strand'] == '+' else row['end'] - 1,
        axis=1,
    )
    return genes_df, exons_df


def load_gene_annotations_for_gene(gtf_path: Path, gene_name: str, shortcut_dir: Path):
    """Load gene annotations for a single gene, preferring cached shortcuts."""
    shortcut_dir = ensure_dir(Path(shortcut_dir))
    gene_file = shortcut_dir / f'{gene_name}_gene_annotations.csv'
    exon_file = shortcut_dir / f'{gene_name}_exon_annotations.csv'

    if gene_file.exists() and exon_file.exists():
        print(f'Loading cached annotations for {gene_name} from {shortcut_dir}')
        genes_df = pd.read_csv(gene_file)
        exons_df = pd.read_csv(exon_file)
        return genes_df, exons_df, True

    print(f'No cached annotations for {gene_name}. Parsing full GTF...')
    genes_df, exons_df = _parse_gtf_for_gene(gtf_path, gene_name)

    genes_df.to_csv(gene_file, index=False)
    exons_df.to_csv(exon_file, index=False)
    print(f'Saved shortcuts: {gene_file.name}, {exon_file.name}')
    return genes_df, exons_df, False


def get_gene_info(gene_name: str, genes_df: pd.DataFrame, exons_df: pd.DataFrame) -> Dict:
    """Extract gene metadata and exon coordinates for the requested gene."""
    mask = genes_df['gene_name'].str.upper() == gene_name.upper()
    if not mask.any():
        raise ValueError(f"Gene '{gene_name}' not found in provided annotations")

    gene_info = genes_df[mask].iloc[0]
    gene_exons = (
        exons_df[exons_df['gene_name'].str.upper() == gene_name.upper()]
        .drop_duplicates(subset=['chrom', 'start', 'end'])
        .reset_index(drop=True)
    )

    result = {
        'gene_name': gene_info['gene_name'],
        'chrom': gene_info['chrom'],
        'start': int(gene_info['start']),
        'end': int(gene_info['end']),
        'strand': gene_info['strand'],
        'tss': int(gene_info['tss']),
        'exons': gene_exons[['chrom', 'start', 'end']].copy(),
    }
    return result


# =============================================================================
# MAIN ANALYSIS
# =============================================================================

SEQ_LEN = 524288
LABEL_LEN = 196608

GTF_CACHE_DIR = PATHS.gtf_cache
GTF_SHORTCUT_DIR = PATHS.gtf_shortcuts
Z_SCORE_OUTPUT_DIR = ensure_dir(PATHS.output_z_score)
PRELIM_OUTPUT_DIR = ensure_dir(PATHS.output_prelim)
REFERENCE_GENOME = 'hg38'

print('Loading gene annotations...')
gtf_path = download_gtf_if_needed(GTF_CACHE_DIR, genome=REFERENCE_GENOME)
genes_df, exons_df, _ = load_gene_annotations_for_gene(
    gtf_path,
    GENE_NAME,
    GTF_SHORTCUT_DIR,
)
gene_info = get_gene_info(GENE_NAME, genes_df, exons_df)
gene_start = gene_info['start']
gene_end = gene_info['end']
gene_tss = gene_info['tss']
gene_len = gene_end - gene_start

print(f'Gene: {GENE_NAME}')
print(f'  Interval: {gene_start:,} - {gene_end:,} ({gene_len:,} bp)')
print(f'  TSS: {gene_tss:,}')

df['snp_pos'] = gene_tss + df['tss_distance']

limit_tss_centered = SEQ_LEN // 2
df['valid_tss_centered'] = df['tss_distance'].abs() <= limit_tss_centered

output_radius = LABEL_LEN // 2


def check_snp_centered(row):
    snp_pos = row['snp_pos']
    window_start = snp_pos - output_radius
    window_end = snp_pos + output_radius

    overlap_start = max(gene_start, window_start)
    overlap_end = min(gene_end, window_end)

    overlap_len = max(0, overlap_end - overlap_start)
    return overlap_len >= (0.5 * gene_len)


df['valid_snp_centered'] = df.apply(check_snp_centered, axis=1)
df['valid_any'] = df['valid_tss_centered'] | df['valid_snp_centered']

n_both = len(df[df['valid_tss_centered'] & df['valid_snp_centered']])
n_tss_only = len(df[df['valid_tss_centered'] & ~df['valid_snp_centered']])
n_snp_only = len(df[~df['valid_tss_centered'] & df['valid_snp_centered']])
n_neither = len(df[~df['valid_tss_centered'] & ~df['valid_snp_centered']])

print(f'\nFeasibility Report (Total SNPs: {len(df)}):')
print(f'  Both methods feasible:       {n_both}')
print(f'  Only TSS-centered feasible:  {n_tss_only}')
print(f'  Only SNP-centered feasible:  {n_snp_only}')
print(f'  Neither feasible:            {n_neither}')

df['z_score'] = df['slope'] / df['slope_se']

sig_snps = df[df['pval_nominal'] < 0.05]
sig_snps_feasible = sig_snps[sig_snps['valid_any']]
pct_sig_feasible = len(sig_snps_feasible) / len(sig_snps) * 100 if len(sig_snps) > 0 else 0

print(f'\nSignificant SNPs (pval < 0.05): {len(sig_snps)}')
print(f'Significant SNPs in Borzoi feasible region: {len(sig_snps_feasible)} ({pct_sig_feasible:.1f}%)')

df_export = df.copy()
exported_variant_count = len(df_export)
print(f'\nGenerating VCF for {exported_variant_count} variants from the full GTEx postfilter set...')


def parse_variant_id(vid):
    parts = vid.split('_')
    chrom = parts[0]
    pos = parts[1]
    ref = parts[2]
    alt = parts[3]
    return chrom, pos, ref, alt


vcf_data = df_export['variant_id'].apply(parse_variant_id).tolist()
vcf_df = pd.DataFrame(vcf_data, columns=['#CHROM', 'POS', 'REF', 'ALT'])
vcf_df['ID'] = df_export['variant_id'].values
vcf_df['QUAL'] = '.'
vcf_df['FILTER'] = 'PASS'
vcf_df['INFO'] = '.'
vcf_df = vcf_df[['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']]

vcf_path = Z_SCORE_OUTPUT_DIR / f'{GENE_NAME}_variants.vcf'
csv_path = Z_SCORE_OUTPUT_DIR / f'{GENE_NAME}_GTEx_z_scores.csv'

with open(vcf_path, 'w') as f:
    f.write('##fileformat=VCFv4.2\n')
    f.write('##source=GTEx_Analysis_v10_eQTL\n')
    f.write('##reference=hg38\n')
    f.write('#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n')

vcf_df.to_csv(vcf_path, mode='a', sep='	', index=False, header=False)
print(f'VCF saved to: {vcf_path}')

df_export.to_csv(csv_path, index=False)
print(f'GTEx Z-scores CSV saved to: {csv_path}')

count_funnel = pd.DataFrame(
    [
        {'step': 'raw_gene_variants_loaded', 'count': raw_variant_count},
        {'step': 'post_af_filter', 'count': post_af_count},
        {'step': 'post_sample_size_filter', 'count': post_sample_size_count},
        {'step': 'exported_z_score_set', 'count': exported_variant_count},
    ]
)
count_funnel_path = PRELIM_OUTPUT_DIR / f'{GENE_NAME}_phase1_count_funnel.csv'
count_funnel.to_csv(count_funnel_path, index=False)
print(f'Count funnel saved to: {count_funnel_path}')

df_export[['variant_id', 'slope', 'slope_se', 'z_score']].head()
